In [ ]:
# importing the libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10 # a popular dataset which consists of 60000 images

In [ ]:
# Datasets and dataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0,1) => normalize(-1,+1)
transform = transforms.Compose([
    transforms.ToTensor(), # Convert the images to tensor data
    transforms.Normalize((0.5, 0.5, 0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|████████████████████████████████████████| 170M/170M [00:33<00:00, 5.05MB/s]


In [ ]:
# dataloader
trainLoader = DataLoader(trainset , batch_size = 64, shuffle=True)
testLoader = DataLoader(testset , batch_size = 64, shuffle=True)

In [ ]:
# Building a CNN model
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        # Convolutional layers
        self.con_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),  # fixed
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(128*4*4,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.con_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [26]:
model = CNN()

In [27]:
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# train the model
epochs = 10

for epoch in range(epochs):
    model.train()   # important
    running_loss = 0.0

    for images, labels in trainLoader: 
       
        optimizer.zero_grad()  
        
        outputs = model(images)   # fixed
        loss = criteria(outputs, labels)  
        
        loss.backward() 
        optimizer.step()  

        running_loss += loss.item()  

    train_loss = running_loss / len(trainLoader)
    print(f"Epoch = {epoch+1}/{epochs}, Loss = {train_loss:.4f}")

Epoch = 1/10, Loss = 0.7473
Epoch = 2/10, Loss = 0.6175
Epoch = 3/10, Loss = 0.5095
Epoch = 4/10, Loss = 0.4154
Epoch = 5/10, Loss = 0.3315
Epoch = 6/10, Loss = 0.2594
Epoch = 7/10, Loss = 0.1960
Epoch = 8/10, Loss = 0.1577
Epoch = 9/10, Loss = 0.1335
Epoch = 10/10, Loss = 0.1062


In [34]:
# Evaluation
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for images, labels in testLoader:
        outputes = model(images)
        _, predicted = torch.max(outputes , 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print("Accurcy: " , correct/total * 100)

Accurcy:  75.17
